# DeepFilterNet3 V2 Fine-Tuning with Defence-Augmented Data on Kaggle T4×2

This notebook performs controlled **V2 fine-tuning** of DeepFilterNet3 on Kaggle GPU using the newly audited `panavpayappagoudar/sih-26-processed-audio` dataset combined with `anc-sih-training-data`.
It loads initial weights from `model.hk` (V2-A continued fine-tuning), balances firearms, drone, vehicle engine, and environmental noise against clean speech, runs 3 full training epochs, exports `model_v2_best.hk` and `model_v2_final.hk`, and performs strict HK round-trip verification.

In [ ]:
# === SETUP: Build deepfilterlib from source (Rust + maturin) for Kaggle Py3.10/3.12 ===
import subprocess, sys, os, types, shutil
from pathlib import Path

def sh(*args, check=True, env=None, cwd=None):
    full_env = os.environ.copy()
    full_env["PATH"] = os.path.expanduser("~/.cargo/bin") + os.pathsep + full_env.get("PATH","")
    if env: full_env.update(env)
    r = subprocess.run(list(args), capture_output=True, text=True, env=full_env, cwd=cwd)
    out = (r.stdout + r.stderr).strip()
    if out: print(out[-3000:])
    if check and r.returncode != 0:
        raise RuntimeError(f"Command {args[0]} failed (exit {r.returncode})")
    return r

# 0. Verify GPU
import torch
assert torch.cuda.is_available(), "Enable GPU (T4 x2) in session settings before running"
print(f"GPU: {torch.cuda.get_device_name(0)} | count: {torch.cuda.device_count()}")

# 1. Ensure Rust/Cargo
print("\n[1/6] Rust toolchain...")
cargo_path = shutil.which("cargo") or os.path.expanduser("~/.cargo/bin/cargo")
if not os.path.isfile(cargo_path):
    print("  Installing rustup...")
    sh("bash", "-c", "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable --profile minimal")
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + os.pathsep + os.environ.get("PATH", "")
sh("cargo", "--version")
print("  Rust: OK")

# 2. Install maturin
print("\n[2/6] Installing maturin...")
sh(sys.executable, "-m", "pip", "-q", "install", "maturin>=1.0,<2.0")
maturin_bin = shutil.which("maturin") or str(Path(sys.executable).parent / "maturin")
sh(maturin_bin, "--version")

# 3. Clone DeepFilterNet v0.5.6 source
print("\n[3/6] Cloning DeepFilterNet v0.5.6 source...")
SRC = Path("/kaggle/temp/DeepFilterNet-src")
if not (SRC / "Cargo.toml").exists():
    sh("git", "clone", "--depth=1", "--branch=v0.5.6", "https://github.com/Rikorose/DeepFilterNet.git", str(SRC))
PYDF = SRC / "pyDF"

# 4. Patch pyDF/pyproject.toml
pptoml = PYDF / "pyproject.toml"
original = pptoml.read_text()
patched = original.replace('requires = ["maturin>=0.11,<0.12"]', 'requires = ["maturin>=1.0,<2.0"]').replace('requires = ["maturin>=0.11,<1.0"]', 'requires = ["maturin>=1.0,<2.0"]')
pptoml.write_text(patched)

# 5. Build libdf wheel
WHEEL_OUT = Path("/kaggle/temp/df_wheels")
WHEEL_OUT.mkdir(parents=True, exist_ok=True)
sh(maturin_bin, "build", "--release", "--strip", "--interpreter", sys.executable, "--out", str(WHEEL_OUT), cwd=str(PYDF))
wheel = sorted(list(WHEEL_OUT.glob("*.whl")))[-1]
sh(sys.executable, "-m", "pip", "-q", "install", str(wheel), "--force-reinstall", "--no-deps")
print("  deepfilterlib: INSTALLED")

# 6. Install remaining Python packages
sh(sys.executable, "-m", "pip", "-q", "install", "deepfilternet==0.5.6", "--no-deps")
sh(sys.executable, "-m", "pip", "-q", "install", "hknt==1.1.0", "soundfile", "scipy", "appdirs", "loguru", "sympy", "packaging>=23.0,<24.0", "requests")

# torchaudio compat shim
try:
    import torchaudio
    if not hasattr(getattr(torchaudio, "backend", None), "common"):
        cm = types.ModuleType("torchaudio.backend.common")
        class AudioMetaData:
            def __init__(self, sr, nf, nc, bps, enc):
                self.sample_rate=sr; self.num_frames=nf; self.num_channels=nc; self.bits_per_sample=bps; self.encoding=enc
        cm.AudioMetaData = AudioMetaData
        sys.modules["torchaudio.backend.common"] = cm
        if not hasattr(torchaudio, "backend"):
            bm = types.ModuleType("torchaudio.backend")
            bm.common = cm
            sys.modules["torchaudio.backend"] = bm
            torchaudio.backend = bm
        else:
            torchaudio.backend.common = cm
    print("torchaudio shim: OK")
except Exception as e:
    print(f"shim note: {e}")

import os, tarfile, csv, random, json as _json, math, time
from pathlib import Path
import numpy as np, soundfile as sf, scipy.signal as signal
import torch.nn as nn, torch.nn.functional as F
import hk
from df.enhance import init_df, df_features, maybe_download_model
print(f"numpy={np.__version__} torch={torch.__version__}")
print("Setup complete.")


In [ ]:
# === INITIALIZE DEEPFILTERNET3 & LOAD V1 CHECKPOINT (V2-A) ===
base = maybe_download_model('DeepFilterNet3')
model, df_state, suffix = init_df(base, log_file=None)
DF_SR = df_state.sr()  # 48000 Hz
print(f'DeepFilterNet3 Native SR={DF_SR}')

OUT = Path('/kaggle/working/dfn3_v2_sih_defence_augmented')
OUT.mkdir(parents=True, exist_ok=True)

# Look for V1 checkpoint to continue fine-tuning (V2-A)
v1_search = [
    Path('/kaggle/input/deepfilternet3-hknt-anc/model.hk'),
    Path('/kaggle/input/anc-sih-training-data/model.hk'),
    Path('/kaggle/input/deepfilternet3_hknt/model.hk'),
] + list(Path('/kaggle/input').glob('*/*.hk')) + list(Path('/kaggle/input').glob('*/*/*.hk'))
v1_candidates = [p for p in v1_search if p.exists()]
if v1_candidates:
    v1_hk_path = v1_candidates[0]
    print(f'Loading V1 checkpoint from {v1_hk_path}')
    v1_state = hk.load_file(str(v1_hk_path))
    model.load_state_dict(v1_state, strict=True)
    print('Loaded V1 checkpoint for V2-A continued fine-tuning!')
else:
    print('No V1 model.hk found in input; starting fine-tuning from pretrained reference.')

model = model.cpu()
hk.save_file(model.state_dict(), str(OUT / 'pretrained_reference.hk'), metadata={'architecture': 'DeepFilterNet3', 'stage': 'v2_initial_weights'})

device = torch.device('cuda:0')
model = model.to(device)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model, device_ids=list(range(torch.cuda.device_count())))
    print(f'DataParallel across {torch.cuda.device_count()} GPUs')



In [ ]:
# === DATASET SETUP & DEFENCE-AUGMENTED BALANCED SAMPLER ===
# Discover mounted Kaggle inputs
SIH_DS = None
for p in Path('/kaggle/input').rglob('metadata.csv'):
    SIH_DS = p.parent
    break
assert SIH_DS is not None, "sih-26-processed-audio dataset not found in /kaggle/input"
print(f"Discovered sih-26-processed-audio at: {SIH_DS}")

# Load sih-26 metadata
with open(SIH_DS / 'metadata.csv', newline='', encoding='utf-8') as f:
    sih_rows = list(csv.DictReader(f))
print(f"Total raw entries in sih-26: {len(sih_rows):,}")

# Categorize defence noise vs speech
gunfire_rows = [r for r in sih_rows if 'firearms' in r.get('dataset_source','').lower() or 'gun' in r.get('internal_path','').lower()]
drone_rows = [r for r in sih_rows if 'drone' in r.get('dataset_source','').lower()]
vehicle_rows = [r for r in sih_rows if 'vehicle' in r.get('dataset_source','').lower()]
env_rows = [r for r in sih_rows if 'demand' in r.get('dataset_source','').lower()]
speech_rows = [r for r in sih_rows if 'english' in r.get('dataset_source','').lower() or 'librispeech' in r.get('dataset_source','').lower()]

print(f"Defence Subsets: Gunfire={len(gunfire_rows)}, Drone={len(drone_rows)}, Vehicle={len(vehicle_rows)}, Env={len(env_rows)}, Speech={len(speech_rows)}")

def read_audio(path, target_sr):
    x, sr = sf.read(str(path), dtype='float32', always_2d=True)
    x = x.mean(axis=1)
    if sr != target_sr:
        x = signal.resample_poly(x, target_sr, sr).astype('float32')
    return x

class V2_MixSet(torch.utils.data.Dataset):
    def __init__(self, speech, gunfire, drone, vehicle, env, count=3600, sr=48000):
        self.speech = speech
        self.gunfire = gunfire
        self.drone = drone
        self.vehicle = vehicle
        self.env = env
        self.count = count
        self.sr = sr
        self.n = sr * 2  # 2-second clips

    def _resolve(self, row):
        src = row.get('dataset_source','')
        ip = row.get('internal_path','')
        p1 = SIH_DS / src / ip
        if p1.exists(): return p1
        p2 = SIH_DS / row.get('filename','')
        if p2.exists(): return p2
        return p1

    def _load_clip(self, path):
        try:
            x = read_audio(path, self.sr)
            if len(x) < self.n:
                x = np.tile(x, math.ceil(self.n / max(1, len(x))))
            start = random.randrange(max(1, len(x) - self.n + 1))
            return x[start:start + self.n]
        except Exception:
            return np.zeros(self.n, dtype='float32')

    def __len__(self):
        return self.count

    def __getitem__(self, i):
        c = self._load_clip(self._resolve(self.speech[i % len(self.speech)]))
        
        # Balanced noise selection: 40% Gunfire, 20% Drone, 20% Vehicle, 20% Env
        r_val = random.random()
        if r_val < 0.40 and self.gunfire:
            n_row = self.gunfire[random.randrange(len(self.gunfire))]
        elif r_val < 0.60 and self.drone:
            n_row = self.drone[random.randrange(len(self.drone))]
        elif r_val < 0.80 and self.vehicle:
            n_row = self.vehicle[random.randrange(len(self.vehicle))]
        elif self.env:
            n_row = self.env[random.randrange(len(self.env))]
        else:
            n_row = self.speech[(i + 1) % len(self.speech)]
            
        n = self._load_clip(self._resolve(n_row))
        
        snr = random.choice([-5, 0, 5, 10, 15, 20])
        c = c / (np.sqrt(np.mean(c**2)) + 1e-8)
        n = n / (np.sqrt(np.mean(n**2)) + 1e-8)
        n = n * (10 ** (-snr / 20))
        noisy = c + n
        noisy = noisy / max(1.0, float(np.max(np.abs(noisy))))
        
        return {
            'clean': torch.from_numpy(c.copy()).unsqueeze(0),
            'noisy': torch.from_numpy(noisy.astype('float32')).unsqueeze(0),
        }

train_ds = V2_MixSet(speech_rows, gunfire_rows, drone_rows, vehicle_rows, env_rows, count=3600, sr=DF_SR)
val_ds   = V2_MixSet(speech_rows, gunfire_rows, drone_rows, vehicle_rows, env_rows, count=256, sr=DF_SR)
print(f"V2 Training set size: {len(train_ds)} mixtures | Val size: {len(val_ds)}")


In [ ]:
# === TRAINING LOOP (3 EPOCHS) ===
def get_nb_df():
    return model.module.nb_df if isinstance(model, nn.DataParallel) else model.nb_df

def batch_df_features(audio_batch, df_state, nb_df):
    specs, erbs, specfs = [], [], []
    for i in range(audio_batch.shape[0]):
        item = audio_batch[i]
        if item.ndim == 1: item = item.unsqueeze(0)
        s, e, sf = df_features(item.cpu(), df_state, nb_df)
        specs.append(s.squeeze(0))
        erbs.append(e.squeeze(0))
        specfs.append(sf.squeeze(0))
    return torch.stack(specs, dim=0), torch.stack(erbs, dim=0), torch.stack(specfs, dim=0)

class AudioAdapter(nn.Module):
    def __init__(self, net):
        super().__init__()
        self.net = net

    def forward(self, noisy_t, clean_t, df_state, device):
        nb_df = get_nb_df()
        spec, erb, specf = batch_df_features(noisy_t, df_state, nb_df)
        target, _, _     = batch_df_features(clean_t, df_state, nb_df)
        spec, erb, specf, target = spec.to(device), erb.to(device), specf.to(device), target.to(device)
        pred = self.net(spec.clone(), erb, specf)[0]
        loss = F.l1_loss(pred.float(), target.float()) + 0.25 * F.l1_loss(torch.linalg.vector_norm(pred.float(), dim=-1), torch.linalg.vector_norm(target.float(), dim=-1))
        return loss

    def save_hk(self, path, stage_name='sih26052_v2_finetuned'):
        net = self.net.module if isinstance(self.net, nn.DataParallel) else self.net
        cpu_state = {k: (v.detach().cpu() if isinstance(v, torch.Tensor) else v) for k, v in net.state_dict().items()}
        hk.save_file(cpu_state, str(path), metadata={'architecture': 'DeepFilterNet3', 'container': 'HK', 'stage': stage_name})
        print(f'Saved HK checkpoint: {path}')

adapter = AudioAdapter(model).to(device)
optimizer = torch.optim.AdamW(adapter.parameters(), lr=1e-5, weight_decay=1e-4)

NUM_EPOCHS = 3
BATCH_SIZE = 16
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_loader   = torch.utils.data.DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2)

history = []
best_val_loss = float('inf')
adapter.train()

for epoch in range(NUM_EPOCHS):
    total_loss = 0.0; seen = 0; t0 = time.time()
    for step, batch in enumerate(train_loader):
        noisy = batch['noisy']; clean = batch['clean']
        optimizer.zero_grad(set_to_none=True)
        loss = adapter(noisy, clean, df_state, device)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(adapter.parameters(), 1.0)
        optimizer.step()
        total_loss += float(loss.detach()); seen += 1
        if (step + 1) % 25 == 0:
            print(f'[epoch {epoch+1}/{NUM_EPOCHS}] step={step+1:4d} loss={total_loss/seen:.5f} elapsed={time.time()-t0:.0f}s', flush=True)
            
    # Validate epoch
    adapter.eval()
    v_loss = 0.0; v_seen = 0
    with torch.no_grad():
        for vb in val_loader:
            vl = adapter(vb['noisy'], vb['clean'], df_state, device)
            v_loss += float(vl) * len(vb['noisy']); v_seen += len(vb['noisy'])
    val_loss = v_loss / max(v_seen, 1)
    adapter.train()
    
    epoch_rec = {'epoch': epoch+1, 'mean_train_loss': total_loss/max(1,seen), 'val_loss': val_loss}
    history.append(epoch_rec)
    print(f'=== Epoch {epoch+1} Complete: Train Loss={total_loss/seen:.5f} | Val Loss={val_loss:.5f} ===', flush=True)
    
    adapter.save_hk(OUT / f'epoch-{epoch+1}.hk', stage_name=f'epoch_{epoch+1}')
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        adapter.save_hk(OUT / 'model_v2_best.hk', stage_name='best_checkpoint')

adapter.save_hk(OUT / 'model_v2_final.hk', stage_name='final_checkpoint')
print('V2 Training Complete.', flush=True)


In [ ]:
# === HK ROUND-TRIP VERIFICATION & REPORT GENERATION ===
best_hk = hk.load_file(str(OUT / 'model_v2_best.hk'))
bare = model.module if isinstance(model, nn.DataParallel) else model
bare.load_state_dict(best_hk, strict=True)
print('model_v2_best.hk HK round-trip check: PASSED STRICTLY')

report = {
    'architecture': 'DeepFilterNet3',
    'container_format': 'HK (hknt 1.1.0)',
    'sample_rate': DF_SR,
    'epochs': NUM_EPOCHS,
    'best_val_loss': best_val_loss,
    'history': history,
    'hk_roundtrip': 'strict load passed',
    'output_files': [
        str(OUT / 'epoch-1.hk'),
        str(OUT / 'epoch-2.hk'),
        str(OUT / 'epoch-3.hk'),
        str(OUT / 'model_v2_best.hk'),
        str(OUT / 'model_v2_final.hk'),
    ],
}
(OUT / 'training_report.json').write_text(_json.dumps(report, indent=2))
print(_json.dumps(report, indent=2))
print(f'\nAll outputs saved to: {OUT}')
